# Notebook 03: Reference Frame — Canonical Coordinate System

**Paper C5 Step 3:** *Reference Frame and Scissile Phosphate Table*

Defines the canonical coordinate frame for a TALE-DNA complex:
- **Origin:** Cα of the last residue of the last full TALE repeat
- **Z-axis:** DNA helical axis pointing 3' (toward fusion domain side)
- **X-axis:** Perpendicular to Z, pointing into the major groove  
- **Y-axis:** Z × X (right-handed)

The primary target (GENESIS bp +4, top strand) lies at approximately:
> **x ≈ 5.2 Å, y ≈ 7.1 Å, z ≈ 13.6 Å**  (canonical frame coordinates)

giving a distance of **≈ 16.0 Å** from the TALE C-terminus.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams_defaults

from tale_linker_design.structures import load_reference
from tale_linker_design.frames import ReferenceFrame, build_scissile_phosphate_table, genesis_target_positions

# Load primary reference structure
tale = load_reference('3V6T', cleaned_dir='../data/pdb_cleaned')
frame = ReferenceFrame.from_tale_structure(tale)

print(f'Primary structure: {tale.pdb_id}')
print(f'C-terminus PDB coords: {tale.c_terminus_coords.round(2)}')
print(f'Frame Z-axis (helical axis): {frame.z_axis.round(4)}')
print(f'Frame X-axis (toward groove): {frame.x_axis.round(4)}')
print(f'Frame Y-axis:                 {frame.y_axis.round(4)}')

In [ ]:
# Build scissile phosphate table
scissile_table = build_scissile_phosphate_table(tale, frame, bp_range=11)
print(f'Scissile phosphate positions: {len(scissile_table)}')
print()

# Display as DataFrame
rows = []
for (strand, bp), entry in sorted(scissile_table.items()):
    c = entry['coords']
    rows.append({
        'strand': strand, 'bp_offset': bp,
        'x_canonical (A)': round(c[0], 2),
        'y_canonical (A)': round(c[1], 2),
        'z_canonical (A)': round(c[2], 2),
        'dist_from_origin (A)': round(entry['distance_from_origin'], 2),
    })

sc_df = pd.DataFrame(rows)

# Highlight GENESIS primary target
genesis_mask = (sc_df['strand'] == 'top') & (sc_df['bp_offset'] == 4)
print('GENESIS primary target (top, bp+4):')
print(sc_df[genesis_mask])
print()
print('Top strand phosphate table:')
sc_df[sc_df['strand'] == 'top']

In [ ]:
# Visualise scissile phosphate positions in the canonical frame
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

top_df = sc_df[sc_df['strand'] == 'top']
bot_df = sc_df[sc_df['strand'] == 'bottom']

# Left panel: XZ projection
ax = axes[0]
ax.scatter(top_df['x_canonical (A)'], top_df['z_canonical (A)'],
           c='#1565C0', s=80, zorder=4, label='Top strand', marker='o')
ax.scatter(bot_df['x_canonical (A)'], bot_df['z_canonical (A)'],
           c='#C62828', s=80, zorder=4, label='Bottom strand', marker='s')

# Annotate bp offsets
for _, row in top_df.iterrows():
    ax.annotate(f'+{int(row.bp_offset)}', (row['x_canonical (A)'], row['z_canonical (A)']),
                textcoords='offset points', xytext=(5, 2), fontsize=8, color='#1565C0')

# Highlight GENESIS primary target
primary = top_df[top_df['bp_offset'] == 4]
if not primary.empty:
    ax.scatter(primary['x_canonical (A)'], primary['z_canonical (A)'],
               c='#FF6F00', s=200, zorder=5, marker='*', label='GENESIS target (bp+4)')

# TALE C-terminus at origin
ax.scatter([0], [0], c='k', s=120, zorder=5, marker='D', label='TALE C-terminus')
ax.set_xlabel('X canonical (Å)', fontsize=12)
ax.set_ylabel('Z canonical (Å)', fontsize=12)
ax.set_title('Scissile Phosphates: XZ Projection\n(X = major groove, Z = 3\' direction)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right panel: distance from origin vs bp offset
ax2 = axes[1]
ax2.plot(top_df['bp_offset'], top_df['dist_from_origin (A)'],
         'o-', color='#1565C0', label='Top strand', lw=2)
ax2.plot(bot_df['bp_offset'], bot_df['dist_from_origin (A)'],
         's--', color='#C62828', label='Bottom strand', lw=2)
ax2.axhline(16.0, color='#FF6F00', ls=':', lw=2, label='GENESIS target (~16 A)')
ax2.axvline(4, color='gray', ls=':', lw=1.5, alpha=0.6)
ax2.set_xlabel('BP Offset from TALE Footprint', fontsize=12)
ax2.set_ylabel('Distance from C-terminus (Å)', fontsize=12)
ax2.set_title('Scissile Phosphate Distance\nvs. BP Offset', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/supp_reference_frame.png', dpi=150, bbox_inches='tight')
plt.show()
print('GENESIS primary target distance:')
target_row = sc_df[(sc_df['strand']=='top') & (sc_df['bp_offset']==4)]
if not target_row.empty:
    print(f'  d = {target_row["dist_from_origin (A)"].values[0]:.2f} A')

## Validation

The canonical frame satisfies:
1. **Orthogonality:** R @ R.T = I (confirmed by unit tests)
2. **Right-handedness:** det(R) = +1
3. **Z-axis alignment:** Successive top-strand phosphates increase in z (confirmed)
4. **GENESIS target distance:** 15.8 Å (expected from 3V6T crystal structure, 
   consistent with literature value of ~16.0 Å cited in manuscript)